# Time Series Anomaly Detection - Student Activity

## Prerequisites

**IMPORTANT:** This activity focuses on time-series-specific anomaly detection methods. For foundational outlier detection methods (IQR, Z-score, Modified Z-score), please complete **Activity 1** first.

---

## Learning Objectives

By the end of this activity, you will be able to:

1. **Understand temporal dependencies** and why time series data requires specialized treatment
2. **Apply resampling techniques** to transform time series data for different analysis granularities
3. **Use time-series visualizations** (lag plots, autocorrelation) to identify temporal patterns and anomalies
4. **Implement the Hampel Filter** for rolling window-based outlier detection
5. **Apply advanced algorithms** (STRAY, Matrix Profile) for concept drift and subsequence anomaly detection
6. **Compare and select** appropriate methods based on the type of anomaly and temporal context

---

## Why Time Series is Different: Temporal Dependence

Before we dive into the recipes, it's crucial to understand why time series data requires special handling compared to standard datasets.

In standard statistical analysis, we often assume data points are **independent and identically distributed (i.i.d.)**. This means the value of one observation doesn't depend on the others.

**Time series data violates this assumption:**

- **Temporal Dependence:** Today's value is often related to yesterday's value (autocorrelation)
- **Trend:** The data may have an upward or downward tendency over time
- **Seasonality:** Patterns may repeat at regular intervals (daily, weekly, yearly)
- **Concept Drift:** Statistical properties may change over time in unforeseen ways
- **Local Context Matters:** An "outlier" might be normal in one time period but abnormal in another

The methods in this activity specifically address these temporal characteristics.

---

## Technical Requirements

In [ ]:
!uv pip install statsmodels seaborn sktime stumpy

In [ ]:
import matplotlib 
import pandas as pd
import scipy 
import statsmodels

print(f'''
matplotlib -> {matplotlib.__version__}
pandas -> {pd.__version__}   
scipy -> {scipy.__version__}
statsmodels -> {statsmodels.__version__}
''')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns

In [ ]:
plt.rcParams["figure.figsize"] = [12, 5]

## Dataset: NYC Taxi Passengers

We'll use the NYC Taxi dataset, which captures the number of taxi passengers at 30-minute intervals from July 1, 2014, to May 31, 2015 (10,320 records).

### Known Anomalies

The dataset contains **five known anomalies** that we can use to evaluate our detection methods:

- **November 1, 2014**: Day before NYC Marathon
- **November 27, 2014**: Thanksgiving Day
- **December 25, 2014**: Christmas Day
- **January 1, 2015**: New Year's Day
- **January 27, 2015**: North American Blizzard (vehicles ordered off streets)

These events represent different types of anomalies:
- **Holiday effects** (reduced traffic)
- **Special events** (altered patterns)
- **Weather emergencies** (extreme disruptions)

In [ ]:
# Load the dataset
file = Path("../data/nyc_taxi.csv")
nyc_taxi = pd.read_csv(file,
                    index_col='timestamp',
                    parse_dates=True)

nyc_taxi.index.freq = '30min'

In [ ]:
# Known anomaly dates
nyc_dates = [
    "2014-11-01",  # NYC Marathon
    "2014-11-27",  # Thanksgiving
    "2014-12-25",  # Christmas
    "2015-01-01",  # New Year
    "2015-01-27"   # Blizzard
]

In [ ]:
# Visualization helper function
def plot_outliers(outliers, data, method='Method', halignment='right', valignment='bottom', labels=False):
    """
    Plot time series data with highlighted outliers.
    
    Parameters
    ----------
    outliers : pandas.DataFrame or pandas.Series
        The DataFrame or Series containing the outlier data points.
    data : pandas.DataFrame or pandas.Series
        The complete time series data.
    method : str
        The outlier detection method used, displayed in the plot title.
    halignment : str
        Horizontal alignment for the date labels ('left', 'center', or 'right').
    valignment : str
        Vertical alignment for the date labels ('top', 'center', or 'bottom').
    labels : bool
        If True, displays date labels for each outlier point.
    """
    
    fig, ax = plt.subplots(figsize=(10, 6))
        
    data.plot(ax=ax, alpha=0.6)
    
    # Plot outliers
    if labels:
        outliers.plot(ax=ax, style='rx', markersize=8, legend=False)
        
        # Add text labels for each outlier
        for idx, value in outliers['value'].items():
            ax.text(idx, value, f'{idx.date()}', 
                   horizontalalignment=halignment, 
                   verticalalignment=valignment)
    else:
        outliers.plot(ax=ax, style='rx', legend=False)
    
    ax.set_title(f'NYC Taxi - {method}')
    ax.set_xlabel('date')
    ax.set_ylabel('# of passengers')
    ax.legend(['nyc taxi', 'outliers'])
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize the full dataset
nyc_taxi.plot(title="NYC Taxi Passengers (30-min intervals)", alpha=0.6)
plt.ylabel('# of passengers')
plt.show()

---

# Recipe 1: Time Series Data Preparation & Resampling

## Introduction

Resampling transforms your time series data by changing its frequency, which has significant implications for outlier detection.

**Key Concepts:**

- **Downsampling**: Reduce frequency (e.g., 30-min → daily) to smooth noise and identify global patterns
- **Upsampling**: Increase frequency (e.g., daily → hourly) to fill gaps or align datasets
- **Aggregation Methods**: Different functions (mean, sum, min, max) reveal different aspects of the data
- **Temporal Granularity**: The right frequency depends on your use case and the nature of anomalies

## Why Resampling Matters for Anomaly Detection

- **Reduces noise**: High-frequency data may have random fluctuations that obscure real anomalies
- **Reveals patterns**: Aggregating to daily/weekly views can make seasonal patterns more obvious
- **Computational efficiency**: Fewer data points speed up complex algorithms
- **Trade-offs**: You may lose fine-grained anomalies when aggregating

## 1.1 Examine the Original Data

In [ ]:
print("First 5 rows:")
print(nyc_taxi.head())
print(f"\nCurrent frequency: {nyc_taxi.index.freq}")
print(f"Total records: {len(nyc_taxi)}")

## 1.2 Your Task: Implement Downsampling Function

### TODO 1: Create a resampling function

Create a function that resamples time series data to different frequencies and aggregation methods.

In [ ]:
def resample_timeseries(df, frequency='D', agg_method='mean'):
    """
    Resample time series data to a different frequency.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Time series data with DatetimeIndex
    frequency : str
        Frequency string (e.g., 'D' for daily, 'W' for weekly, 'ME' for month-end)
    agg_method : str
        Aggregation method: 'mean', 'sum', 'min', 'max', 'median'
    
    Returns
    -------
    pandas.DataFrame
        Resampled time series
    """
    # TODO: Implement the resampling logic
    # Hint: Use df.resample(frequency) followed by the aggregation method
    # Example: df.resample('D').mean()
    
    pass  # Replace with your implementation

## 1.3 Test Your Function

In [ ]:
# Test with daily mean (this is the standard resampling we'll use)
tx = resample_timeseries(nyc_taxi, frequency='D', agg_method='mean')
print(f"Resampled to daily: {len(tx)} records")
print(tx.head())

### Exercise 1.1: Experiment with Different Aggregations

Try different aggregation methods and observe how they affect the detection of anomalies.

In [ ]:
# TODO: Try resampling to daily with 'min' and 'max'
# Which aggregation would be most sensitive to anomalies?
# Why might 'sum' be problematic for anomaly detection?

# Your code here


### Question 1: Which aggregation method do you think would be best for detecting:
- (a) Days with abnormally LOW passenger counts (like the blizzard)?
- (b) Days with abnormally HIGH passenger counts?
- (c) Overall daily anomalies?

**Your answer:**
```
Write your reasoning here...
```

## 1.4 Visualize Known Outliers on Resampled Data

In [ ]:
# Create daily mean resampled data (standard for this activity)
tx = nyc_taxi.resample('D').mean()
known_outliers = tx.loc[nyc_dates]

plot_outliers(known_outliers, tx, 'Known Outliers', labels=True)

### Exercise 1.2: Multiple Aggregation Analysis

Use `.agg()` to compute multiple statistics simultaneously.

In [ ]:
# TODO: Resample to monthly ('ME') with multiple aggregations
# Include: mean, min, max, median, std
# Look at the results - which months show suspicious min values?

# Your code here


---

# Recipe 2: Visual Exploration for Time Series

## Introduction

While basic visualizations (histograms, box plots) are covered in Activity 1, **time series data requires specialized visualizations** that capture temporal relationships.

**Time Series-Specific Visualizations:**

1. **Lag Plots**: Show relationship between current and previous values
2. **Autocorrelation Functions (ACF)**: Quantify temporal dependencies
3. **Rolling Statistics**: Visualize changing baselines and variance

These reveal:
- **Autocorrelation**: How strongly today's value relates to yesterday's
- **Temporal patterns**: Cyclic or seasonal behaviors
- **Outliers that break patterns**: Values that disrupt the temporal structure

## 2.1 Lag Plots

A lag plot shows each data point plotted against its previous value (by default with lag=1). Points that fall far from the main cluster often represent anomalous shifts in the time series.

In [ ]:
from pandas.plotting import lag_plot

# Default lag=1 (each point vs. previous point)
lag_plot(tx['value'], lag=1)
plt.title('Lag Plot (lag=1) - NYC Taxi Daily Passengers')
plt.show()

### What to Look For:

- **Diagonal clustering**: Indicates strong positive autocorrelation
- **Scattered points**: Weak or no autocorrelation
- **Points far from cluster**: Potential anomalies or structural breaks

### TODO 2: Create lag plots for different lags

In [ ]:
# TODO: Create a figure with 4 subplots showing lag plots for lag=1, 7, 14, 30
# Hint: Use plt.subplots(2, 2, figsize=(12, 10))
# For daily data, lag=7 shows weekly patterns, lag=30 shows monthly patterns

# Your code here


### Question 2: 
- Which lag value shows the strongest correlation?
- What does this tell you about the temporal structure of NYC taxi data?
- Can you visually identify any outliers in the lag plots?

**Your answer:**
```
Write your interpretation here...
```

## 2.2 Autocorrelation Function (ACF)

The ACF plot shows correlation between the time series and lagged versions of itself at different time lags. This helps identify:
- **Periodic patterns** (e.g., weekly seasonality)
- **Strength of temporal dependence**
- **Appropriate window sizes** for methods like the Hampel Filter

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Autocorrelation Function
plot_acf(tx['value'], lags=40, ax=axes[0])
axes[0].set_title('Autocorrelation Function (ACF)')

# Partial Autocorrelation Function
plot_pacf(tx['value'], lags=40, ax=axes[1])
axes[1].set_title('Partial Autocorrelation Function (PACF)')

plt.tight_layout()
plt.show()

### Interpretation:

- **ACF**: Shows total correlation at each lag (direct + indirect)
- **PACF**: Shows only direct correlation at each lag
- **Blue shaded area**: Confidence interval (values outside are statistically significant)
- **Periodic spikes**: Indicate seasonality (e.g., weekly patterns at lag 7)

### Exercise 2.1: Analyze Autocorrelation

Based on the ACF plot:
1. Identify the lags with significant autocorrelation
2. Do you see evidence of weekly seasonality?
3. What window size would you recommend for the Hampel Filter? (We'll use this in Recipe 3)

**Your analysis:**
```
Write your observations here...
```

## 2.3 Rolling Statistics Visualization

Rolling (moving) statistics help visualize how the time series properties change over time.

In [ ]:
# Calculate rolling statistics
window = 7  # 7-day window

rolling_mean = tx['value'].rolling(window=window).mean()
rolling_std = tx['value'].rolling(window=window).std()

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Original data with rolling mean
axes[0].plot(tx.index, tx['value'], label='Original', alpha=0.5)
axes[0].plot(tx.index, rolling_mean, label=f'{window}-day Rolling Mean', color='red')
axes[0].set_title('NYC Taxi Passengers with Rolling Mean')
axes[0].set_ylabel('Passengers')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Rolling standard deviation
axes[1].plot(tx.index, rolling_std, color='orange')
axes[1].set_title(f'{window}-day Rolling Standard Deviation')
axes[1].set_ylabel('Std Dev')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### TODO 3: Detect anomalies using rolling statistics

In [ ]:
# TODO: Implement a simple rolling z-score anomaly detector
# 1. Calculate rolling mean and rolling std (7-day window)
# 2. Calculate z-score: (value - rolling_mean) / rolling_std
# 3. Flag points where |z-score| > 2.5 as anomalies
# 4. Visualize the detected anomalies

# Your code here


### Question 3:
How does the rolling z-score method compare to a global z-score (from Activity 1)?
Why might rolling statistics be better for time series data?

**Your answer:**
```
Write your reasoning here...
```

---

# Recipe 3: Hampel Filter (Rolling Window Method)

## Introduction

The Hampel Filter extends the Modified Z-Score concept (from Activity 1) into a **rolling window implementation**. Unlike global methods, the Hampel Filter considers the **local behavior** of the time series.

**Key Advantages:**
- Effective for data with **changing baselines**
- Handles **seasonal patterns** and trends
- Robust to outliers (uses median instead of mean)
- Adaptable to local context

## How It Works

For each data point:
1. Select a window of observations centered on the current point
2. Calculate the **median** of values within this window
3. Calculate the **Median Absolute Deviation (MAD)**
4. Compute a modified z-score: `|x - median| / (k * MAD)`
5. Flag as outlier if z-score > threshold (typically 2.5-3)

Where:
- **k = 1.4826**: Scale factor to make MAD comparable to standard deviation for Gaussian data
- **window_length**: Size of the sliding window
- **n_sigma**: Threshold multiplier (controls sensitivity)

## 3.1 Understanding the Hampel Filter Parameters

Two key parameters influence behavior:

1. **window_length**: Controls the size of the sliding window
   - Larger windows: Detect outliers against broader patterns
   - Smaller windows: More sensitive to recent changes
   - Rule of thumb: Use a window that covers one seasonal cycle (e.g., 7 days for weekly patterns)

2. **n_sigma**: The threshold multiplier
   - Higher values: More conservative (fewer false positives)
   - Lower values: More sensitive (may flag more anomalies)
   - Common range: 2.5 to 3.5

## 3.2 Install and Import

In [ ]:
from sktime.transformations.series.outlier_detection import HampelFilter

## 3.3 Your Task: Implement Hampel Filter Function

### TODO 4: Complete the Hampel outlier detection function

In [ ]:
def hampel_outlier_detection(df, window_length=10, n_sigma=3):
    """
    Detect outliers using sktime's HampelFilter implementation.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Time series data with a 'value' column
    window_length : int
        Size of the sliding window (must be odd)
    n_sigma : float
        Number of standard deviations to use as threshold
    
    Returns
    -------
    tuple
        (outliers DataFrame, transformed DataFrame with filtered values)
    """
    # Create a copy of the input data
    data = df.copy()
    
    # TODO: Initialize the HampelFilter with the given parameters
    # Hint: Use HampelFilter(window_length=..., n_sigma=..., k=1.4826, return_bool=False)
    hampel = None  # Replace with your initialization
    
    # TODO: Apply the filter to transform the data
    # Hint: Use hampel.fit_transform(data)
    transformed = None  # Replace with your transformation
    
    # TODO: Find the outliers by comparing original and transformed data
    # Outliers are marked as NaN in the transformed data
    # Hint: Use data.isna() or data.isnull() to find NaN values
    outlier_mask = None  # Replace with your mask
    outliers = None  # Extract outliers from original data using the mask
    
    return outliers, transformed

## 3.4 Apply the Hampel Filter

In [ ]:
# Start with reasonable default parameters
window_length = 21  # 3-week window
n_sigma = 2.5

outliers_hampel, transformed = hampel_outlier_detection(tx, window_length, n_sigma)

print(f"Detected {len(outliers_hampel)} outliers:")
print(outliers_hampel['value'])

In [ ]:
# Visualize detected outliers
plot_outliers(outliers_hampel[['value']], tx, f'Hampel Filter (window={window_length}, sigma={n_sigma})', labels=True)

## 3.5 Exercise: Parameter Tuning

### TODO 5: Experiment with different parameters

In [ ]:
# TODO: Try at least 3 different combinations of (window_length, n_sigma)
# Compare the results. Which combination:
# - Captures the most known anomalies?
# - Has the fewest false positives?
# - Provides the best balance?

# Suggestions to try:
# - (7, 2.5)   # Short window, moderate threshold
# - (21, 3.0)  # Medium window, conservative threshold
# - (30, 2.5)  # Long window, moderate threshold

# Your code here


### Question 4:
Based on your experiments:
1. How does window_length affect the detected outliers?
2. How does n_sigma affect sensitivity?
3. Which parameter combination would you recommend for this dataset and why?

**Your analysis:**
```
Write your findings here...
```

## 3.6 Imputation (Optional Exploration)

The Hampel Filter replaces outliers with NaN values. We can impute these using various strategies.

In [ ]:
from sktime.transformations.series.impute import Imputer

# Linear imputation
imputer = Imputer(method="linear")
y_corrected = imputer.fit_transform(transformed['filtered'])

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Original
tx['value'].plot(ax=axes[0], title='Original Data', alpha=0.7)

# After Hampel (with NaNs)
transformed['filtered'].plot(ax=axes[1], title='After Hampel Filter (NaNs for outliers)', alpha=0.7)

# After imputation
y_corrected.plot(ax=axes[2], title='After Linear Imputation', alpha=0.7)

plt.tight_layout()
plt.show()

---

# Recipe 4: STRAY (Search TRace AnomalY)

## Introduction

STRAY is designed for detecting anomalies in data streams that exhibit **concept drift** - where statistical properties change over time in unforeseen ways.

**What Makes STRAY Special:**
- Extends HDoutliers algorithm with extreme value theory
- Handles trends and seasonality
- Designed for **non-stationary** time series
- Can detect clusters of anomalies

**Use Cases:**
- Consumer behavior shifts
- Market dynamics changes
- Sensor drift
- Evolving system characteristics

**Parameters:**
- `k`: Number of nearest neighbors (controls sensitivity)
- `alpha`: Significance level for anomaly threshold (e.g., 0.05 = 5% FPR)

## 4.1 Apply STRAY

STRAY is implemented in sktime and relatively straightforward to use.

In [ ]:
from sktime.detection.stray import STRAY

# Initialize and fit the model
model = STRAY(k=7, alpha=0.05)
model.fit(tx['value'])

# Transform returns True for anomalies, False otherwise
output = model.transform(tx['value'])

print(f"Total anomalies detected: {output.sum()}")

In [ ]:
# Extract outliers
outliers_stray = tx[output]
print("Detected anomalies:")
print(outliers_stray)

In [ ]:
# Visualize
plot_outliers(outliers_stray, tx, 'STRAY Anomaly Detection', labels=True)

## 4.2 Compare with Known Anomalies

In [ ]:
# Check how many known anomalies were detected
detected_known = outliers_stray.index.intersection(pd.DatetimeIndex(nyc_dates))
print(f"Known anomalies detected: {len(detected_known)} out of {len(nyc_dates)}")
print("Dates:", list(detected_known.date))

print("\nAll known anomalies:")
print(nyc_dates)

## 4.3 Exercise: Parameter Tuning

### TODO 6: Experiment with STRAY parameters

In [ ]:
# TODO: Try different values of k and alpha
# k suggestions: 3, 5, 7, 10, 15
# alpha suggestions: 0.01, 0.05, 0.1
#
# For each combination:
# 1. Run STRAY
# 2. Count total anomalies detected
# 3. Count how many known anomalies were captured
# 4. Visualize the results
#
# Create a summary table of your findings

# Your code here


### Question 5:
1. How does the `k` parameter (number of neighbors) affect the results?
2. How does `alpha` (significance level) change the number of detected anomalies?
3. Did STRAY detect any anomalies that the Hampel Filter missed? Why might this be?

**Your analysis:**
```
Write your observations here...
```

### Exercise 4.1: Extended Effects

Notice that STRAY might detect dates adjacent to known anomalies (e.g., Dec 26 after Christmas, Jan 26 before the blizzard). 

What does this tell you about how events affect time series data? Is this a feature or a bug?

**Your reflection:**
```
Write your thoughts here...
```

---

# Recipe 5: Matrix Profile

## Introduction

Matrix Profile is fundamentally different from all previous methods. Instead of analyzing individual points, it **analyzes subsequence patterns**.

**Key Concept:**
- A **subsequence** is a continuous segment of fixed length (e.g., 30 consecutive days)
- Matrix Profile compares every subsequence against all other subsequences
- The **matrix profile value** for each subsequence is the distance to its nearest neighbor
- High values = unusual patterns (no similar patterns elsewhere)

**Comparison:**
- **Point-based methods** (z-score, Hampel, IQR, STRAY): Focus on individual values
- **Matrix Profile**: Analyzes pattern similarity over windows

**Strengths:**
- Detects pattern anomalies (not just value anomalies)
- Finds motifs (repeated patterns) and discords (unique patterns)
- Parameter-free distance calculation

**Limitations:**
- Computationally intensive for large datasets
- Requires choosing subsequence length `m`
- May miss single-point anomalies

## 5.1 Understanding Subsequence Length

The `m` parameter (subsequence length) is crucial:

- **Too small**: Captures noise, misses patterns
- **Too large**: Averages out interesting features
- **Rule of thumb**: 
  - For daily data with weekly patterns: m = 7-14
  - For patterns spanning a month: m = 20-30
  - Generally: m should span the characteristic timescale of your domain

## 5.2 Apply Matrix Profile

In [ ]:
from sktime.transformations.panel.matrix_profile import MatrixProfile

# Set the subsequence length (window size)
subsequence_length = 30

# Create the MatrixProfile transformer
mp_transformer = MatrixProfile(m=subsequence_length)
mp_result = mp_transformer.fit_transform(tx)

# The points with highest matrix profile values are the most anomalous
# Use a percentile threshold (e.g., top 5%)
threshold = np.percentile(mp_result, 95)
anomalies = mp_result > threshold

anomalies_series = anomalies.iloc[0]
anomaly_indices = np.where(anomalies_series)[0]
anomaly_timestamps = tx.index[anomaly_indices]

outliers_mp = tx.loc[anomaly_timestamps]
print(f"Matrix Profile detected {len(outliers_mp)} anomalies:")
print(outliers_mp)

In [ ]:
# Compare with known anomalies
detected_known = outliers_mp.index.intersection(pd.DatetimeIndex(nyc_dates))
print(f"Known anomalies detected: {len(detected_known)} out of {len(nyc_dates)}")
print("Dates:", list(detected_known.date))

In [ ]:
# Visualize
plot_outliers(outliers_mp, tx, f'Matrix Profile (m={subsequence_length})', labels=True)

## 5.3 Visualize Matrix Profile Values

Let's visualize the matrix profile values themselves to understand which patterns are most unusual.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Original time series
axes[0].plot(tx.index, tx['value'], alpha=0.7)
axes[0].scatter(outliers_mp.index, outliers_mp['value'], color='red', s=50, zorder=5, label='Anomalies')
axes[0].set_ylabel('Passengers')
axes[0].set_title('NYC Taxi Passengers')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Matrix profile values
axes[1].plot(tx.index, mp_result.values[0], color='orange', alpha=0.7)
axes[1].axhline(y=threshold, color='red', linestyle='--', label=f'95th percentile threshold')
axes[1].set_ylabel('Matrix Profile Value')
axes[1].set_xlabel('Date')
axes[1].set_title('Matrix Profile Values (Higher = More Anomalous)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5.4 Your Task: Parameter Tuning

### TODO 7: Experiment with different subsequence lengths and thresholds

In [ ]:
# TODO: Create a function to run Matrix Profile with different parameters
# Test at least 3 different subsequence lengths: e.g., 7, 14, 30
# Test at least 2 different percentile thresholds: e.g., 90, 95, 99
#
# For each combination:
# 1. Run Matrix Profile
# 2. Count total anomalies detected
# 3. Count known anomalies captured
# 4. Visualize results
#
# Create a comparison summary

def run_matrix_profile(data, m, percentile_threshold):
    """
    Run Matrix Profile with given parameters.
    
    Parameters
    ----------
    data : pandas.DataFrame
        Time series data
    m : int
        Subsequence length
    percentile_threshold : float
        Percentile for anomaly threshold (e.g., 95)
    
    Returns
    -------
    pandas.DataFrame
        Detected outliers
    """
    # TODO: Implement this function
    pass

# Your experimentation code here


### Question 6:
1. How does the subsequence length `m` affect the detected anomalies?
2. What does it mean when Matrix Profile detects a sequence of consecutive days as anomalous?
3. Did Matrix Profile identify any anomalies that other methods missed? Why?

**Your analysis:**
```
Write your interpretation here...
```

## 5.5 Exercise: Identify Pattern Anomalies

Matrix Profile might detect periods like early August that other methods missed. Let's investigate why.

In [ ]:
# TODO: 
# 1. Extract a 30-day window around one of the Matrix Profile anomalies
# 2. Visualize this window
# 3. Compare it to "normal" 30-day windows from other periods
# 4. Can you identify what makes this pattern unusual?

# Hint: Use tx.loc['2014-08-09':'2014-09-09'] to extract a window

# Your code here


**Your findings:**
```
Describe what makes this pattern anomalous...
```

---

# Method Comparison & Selection Guide

## Compare All Methods on Known Anomalies

### TODO 8: Create a comprehensive comparison

In [ ]:
# TODO: Create a comparison table showing:
# - Method name
# - Total anomalies detected
# - Known anomalies detected (count and which ones)
# - False positive rate (approximate)
# - Computational complexity (qualitative: Fast/Medium/Slow)
#
# Present as a pandas DataFrame

# Your code here


## When to Use Each Method

### TODO 9: Complete this decision guide based on your experiments

| Method | Best For | Key Strengths | Limitations | Recommended When... |
|--------|----------|---------------|-------------|--------------------|
| **Hampel Filter** | ??? | ??? | ??? | ??? |
| **STRAY** | ??? | ??? | ??? | ??? |
| **Matrix Profile** | ??? | ??? | ??? | ??? |

**Instructions:** Fill in the table based on your experimentation and understanding.

## Visualization: Create a Comparison Plot

In [ ]:
# TODO: Create a single figure with 4 subplots:
# 1. Original data with known anomalies
# 2. Hampel Filter results
# 3. STRAY results  
# 4. Matrix Profile results
#
# This visual comparison will help identify consensus anomalies vs method-specific detections

# Your code here


---

# Challenge Exercise: Apply to a Different Dataset

## Option A: Airline Passengers Dataset

Apply the methods to the classic airline passengers dataset with known seasonality.

In [ ]:
# Load airline passengers data
# You can use: from sktime.datasets import load_airline
# Or load from seaborn: sns.load_dataset('flights')

# TODO:
# 1. Load and visualize the data
# 2. Apply at least 2 methods (Hampel and one other)
# 3. Compare results
# 4. Discuss whether the detected "anomalies" make sense

# Your code here


## Option B: Generate Synthetic Data

Create your own time series with controlled anomalies.

In [ ]:
# TODO: Generate a synthetic time series with:
# - Trend component
# - Seasonal component
# - Random noise
# - 3-5 deliberately injected anomalies (different types)
#
# Then:
# 1. Apply all three methods
# 2. Evaluate which method detects which type of anomaly best
# 3. Analyze why certain methods work better for certain anomaly types

# Your code here


---

# Reflection Questions

## 1. Temporal Context and Anomalies

**Question:** In the NYC Taxi data, the day before the blizzard (Jan 26) showed unusual patterns. 
- Is this an anomaly or a precursor/early warning?
- How should anomaly detection systems handle such "edge effects"?
- What are the implications for real-time monitoring systems?

**Your reflection:**
```
Write your thoughts here...
```

---

## 2. Concept Drift

**Question:** STRAY is designed for concept drift. 
- What types of concept drift might occur in taxi data over multiple years?
- How would a rolling window method (Hampel) vs. a global method (from Activity 1) handle gradual drift differently?
- When would you need to retrain or recalibrate your anomaly detector?

**Your reflection:**
```
Write your thoughts here...
```

---

## 3. Ensemble Approach

**Question:** Different methods detected different anomalies.
- How could you combine multiple methods into an ensemble detector?
- Would you use voting (majority agreement)? Weighted scores? Union vs. intersection?
- What are the trade-offs between these approaches?

**Your reflection:**
```
Write your thoughts here...
```

---

## 4. Point vs. Pattern Anomalies

**Question:** Matrix Profile detects pattern anomalies while other methods focus on point anomalies.
- Give examples of real-world scenarios where pattern anomalies are more important than point anomalies
- Give examples where point anomalies are more critical
- How would you decide which to prioritize in a production system?

**Your reflection:**
```
Write your thoughts here...
```

---

## 5. Parameter Sensitivity

**Question:** All methods have parameters that significantly affect results.
- In a production environment, how would you systematically tune these parameters?
- What if you don't have labeled anomalies for validation?
- How might you use domain knowledge to guide parameter selection?

**Your reflection:**
```
Write your thoughts here...
```

---

## 6. False Positives vs. False Negatives

**Question:** Consider different application domains:
- **Fraud detection**: Missing fraud (false negative) vs. flagging legitimate transactions (false positive)
- **Equipment monitoring**: Missing equipment failure vs. unnecessary maintenance
- **Healthcare**: Missing critical health events vs. alert fatigue

How would you adjust your method selection and threshold settings for each?

**Your reflection:**
```
Write your thoughts here...
```

---

# Summary and Key Takeaways

## What You've Learned

1. **Temporal Dependencies**: Time series data requires methods that account for autocorrelation, trends, and seasonality

2. **Rolling Window Approaches**: The Hampel Filter demonstrates how local context improves outlier detection in non-stationary data

3. **Concept Drift Detection**: STRAY handles evolving data distributions, crucial for real-world streaming data

4. **Pattern vs. Point Anomalies**: Matrix Profile reveals that not all anomalies are extreme values - unusual patterns matter too

5. **No Single Best Method**: Different methods excel at different anomaly types; understanding trade-offs is essential

6. **Parameter Tuning Matters**: Systematic experimentation and domain knowledge guide effective parameter selection

## Next Steps

- **Activity 3** (if available): Machine learning-based anomaly detection (Isolation Forest, Autoencoders, etc.)
- **Explore deep learning methods**: LSTMs, Transformer models for sequence anomaly detection
- **Study online/streaming algorithms**: For real-time anomaly detection
- **Read research papers**: On the specific algorithms (STRAY, Matrix Profile) for deeper understanding

## Additional Resources

- **Matrix Profile**: https://www.cs.ucr.edu/~eamonn/MatrixProfile.html
- **STRAY Paper**: Talagala et al. (2021) "Anomaly Detection in High Dimensional Data"
- **sktime Documentation**: https://www.sktime.net/
- **Time Series Anomaly Detection Survey**: https://arxiv.org/abs/2106.00134

---

**Congratulations on completing this activity!**